# 🚀 Training Model Versi 7 (V7) di Google Colab (NVIDIA T4 / A100 GPU)
### Pusat Gadai Indonesia — Smartphone Defect Inspection
Notebook ini dikonfigurasi untuk menjalankan pelatihan **Model V7** secara cepat menggunakan akselerasi GPU Cloud (NVIDIA T4 / V100 / A100).

In [ ]:
# 1. Verifikasi Akselerasi GPU NVIDIA
!nvidia-smi

In [ ]:
# 2. Clone Repository Proyek dari GitHub
!git clone https://github.com/Mufti129/defect.git
%cd defect

In [ ]:
# 3. Instalasi Dependensi Ultralytics & PyTorch GPU
!pip install -q ultralytics albumentations opencv-python-headless
import torch
print("PyTorch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

In [ ]:
# 4. Jalankan Pelatihan Model V7 dengan GPU CUDA
# Di GPU T4 Colab: Gunakan batch 8 untuk kecepatan maksimal & stabilitas VRAM
!python train_v7_detector.py --epochs 40 --batch 8 --imgsz 800

In [ ]:
# 5. Cek Live Status & Hasil Evaluasi Metrics Sementara
import json
import pandas as pd
from IPython.display import display

# Baca Live Status JSON
status_file = "weights_v7/training_live_status.json"
try:
    with open(status_file) as f:
        status_data = json.load(f)
    print("=== LIVE STATUS V7 ===")
    print(f"Epoch: {status_data.get('current_epoch')}/{status_data.get('total_epochs')}")
    print(f"Batch: {status_data.get('current_batch')}/{status_data.get('total_batches_per_epoch')} ({status_data.get('epoch_progress_percent')}%) - Overall: {status_data.get('overall_progress_percent')}%")
except Exception as e:
    print("Belum ada file live status:", e)

# Tampilkan Tabel Metrik Evaluasi
csv_file = "weights_v7/results.csv"
try:
    df = pd.read_csv(csv_file)
    display(df.tail(5))
except Exception as e:
    print("Results CSV belum tersedia:", e)

In [ ]:
# 6. Lokasi Penyimpanan Model Terbaik & Download Langsung ke Laptop
import os
from google.colab import files

best_weight = "runs_v7_training/defect_v7_model/weights/best.pt"
if not os.path.exists(best_weight):
    best_weight = "weights_v7/phone_defect_model_v7_best.pt"

if os.path.exists(best_weight):
    print(f"✅ Model terbaik ditemukan di: {best_weight} ({os.path.getsize(best_weight)/1e6:.1f} MB)")
    files.download(best_weight)
    print("Mulai mengunduh file bobot terbaik ke laptop Anda...")
else:
    print("❌ File bobot belum ditemukan. Pastikan minimal Epoch 1 telah selesai.")

In [ ]:
# 7. (Opsional) Backup Otomatis ke Google Drive Pribadi
# Jalankan cell ini jika ingin bobot tersimpan aman di Google Drive Anda
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dest = '/content/drive/MyDrive/Model_V7_PGI/'
os.makedirs(drive_dest, exist_ok=True)

if os.path.exists(best_weight):
    shutil.copy(best_weight, os.path.join(drive_dest, 'phone_defect_model_v7_best.pt'))
    print(f"✅ Bobot berhasil dicadangkan ke Google Drive: {drive_dest}")